In [1]:
import os, time, yaml
from pyspark.sql import SparkSession, Window, functions as F

with open("/home/jovyan/work/config.yaml") as f:
    cfg = yaml.safe_load(f)

HDFS = cfg["hdfs_base"]                              # hdfs://namenode:9000
RAW = HDFS + cfg["paths"]["raw"]                     # zona raw: el CSV tal cual
CURATED = HDFS + cfg["paths"]["curated"]             # zona curada: Parquet limpio
LANG_INFO = HDFS + cfg["paths"]["language_info"]     # tabla auxiliar de idiomas
RESULTS = HDFS + cfg["paths"]["results"]             # zona de resultados

# Igual que en el Tema 5: actuamos como root en HDFS, ANTES de crear la sesión
os.environ["HADOOP_USER_NAME"] = "root"

spark = (SparkSession.builder
         .appName("wikirank-experiments")
         .master(cfg["spark"]["master"])                             # en clase: local[*]
         .config("spark.driver.host", cfg["spark"]["driver_host"])
         .config("spark.executor.memory", cfg["spark"]["executor_memory"])
         .config("spark.hadoop.dfs.replication", cfg["spark"]["dfs_replication"])
         .getOrCreate())
spark


In [2]:
articles = spark.read.parquet(CURATED)
lang_info = spark.read.parquet(LANG_INFO)
schema = ("Wikipedia_article_URL STRING, Language STRING, Page_ID LONG, "
          "Title STRING, WikiRank_score DOUBLE, WikiRank_URL STRING")
raw = spark.read.schema(schema).option("header", True).option("escape", '"').csv(RAW)

In [4]:
import csv
import time

lines = spark.sparkContext.textFile(RAW)
header = lines.first()

def parse(line):
    # Como l.split(","), pero respetando las comas que hay dentro de algunos títulos
    f = next(csv.reader([line]))
    try:
        lang, score = f[1].strip().lower(), float(f[4])
    except (IndexError, ValueError):
        return []                                        # fila rota: se descarta
    return [(lang, score)] if lang and 0 <= score <= 100 else []

t = time.time()
result = (lines.filter(lambda l: l != header)            # quitar la cabecera
               .flatMap(parse)                           # MAP: (idioma, score)
               .mapValues(lambda s: (s, 1))              # (idioma, (score, 1))
               .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))   # SHUFFLE + REDUCE
               .mapValues(lambda x: (x[1], round(x[0] / x[1], 2)))    # (idioma, (n, media))
               .sortBy(lambda kv: -kv[1][0])
               .collect())
print(f"RDD: {time.time() - t:.1f} s")
for lang, (n, avg) in result[:10]:
    print(lang, n, avg)


RDD: 327.0 s
en 6860430 24.71
de 2930971 18.63
fr 2626210 16.12
sv 2591505 20.02
nl 2163715 12.73
ru 1991787 18.18
es 1905093 18.77
it 1875385 13.63
pl 1624175 17.03
zh 1433026 20.31


In [5]:
spark.stop()